# Geometry-V1 Batch 2B operational preflight

**PREPARED_NOT_EXECUTED.** User-only; no current push, Colab, Drive, model, or scientific execution authorization. The proposed independent Drive root is pending user confirmation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, zipfile
REPO_URL='https://github.com/RICHAAARC/CEG-WM.git'; BRANCH='Geometry-V1'
EXECUTION_EXACT='30c04f98e7a6b30e58f3d105412ef534e6742deb'; RUN_ID='geometry-v1-b2b-30c04f98e7a6-operational-01'
RUNNER_MODULE='experiments.run_geometry_v1_qk_operational_preflight'
DRIVE_ROOT=pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/Batch2B')  # pending confirmation
repo=pathlib.Path('/content/geometry-v1-source'); run_dir=DRIVE_ROOT/RUN_ID
if repo.exists() or run_dir.exists(): raise FileExistsError('create-only destination exists')
subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REPO_URL,str(repo)],check=True)
subprocess.run(['git','checkout','--detach',EXECUTION_EXACT],cwd=repo,check=True)
def verify_checkout():
    head=subprocess.run(['git','rev-parse','HEAD'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip()
    clean=subprocess.run(['git','status','--porcelain'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip()
    if head != EXECUTION_EXACT or clean: raise RuntimeError('execution checkout identity differs')
verify_checkout(); subprocess.run([sys.executable,'-m','pip','install',str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL,timeout=600); verify_checkout()


In [ ]:
# Execute only after fresh explicit user authorization.
from google.colab import files, userdata
uploaded=files.upload()
if len(uploaded) not in (1,2): raise ValueError('upload exactly one or two ordinary RGB images')
input_dir=pathlib.Path('/content/geometry-v1-inputs'); input_dir.mkdir(exist_ok=False); paths=[]
for name,data in uploaded.items():
    path=input_dir/pathlib.Path(name).name; path.write_bytes(data); paths.append(str(path))
root_key=userdata.get('CEG_WM_ROOT_KEY'); hf_token=userdata.get('HF_TOKEN')
if not isinstance(root_key,str) or not root_key or not isinstance(hf_token,str) or not hf_token: raise RuntimeError('required Colab secret unavailable')
child_env={n:v for n,v in os.environ.items() if all(m not in n.upper() for m in ('TOKEN','KEY','SECRET','PASSWORD','CREDENTIAL'))}
child_env['CEG_WM_ROOT_KEY']=root_key; child_env['HF_TOKEN']=hf_token; root_key=hf_token=''
try:
    proc=subprocess.run([sys.executable,'-m',RUNNER_MODULE,'--repo-root',str(repo),'--expected-exact',EXECUTION_EXACT,*paths],cwd=repo,env=child_env,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL,timeout=1800,check=False)
    raw=proc.stdout[:4096]
    if proc.returncode != 0 or len(proc.stdout)>4096: raise RuntimeError('sanitized runner failure')
    line=raw.decode('utf-8','strict').strip()
    if not line.startswith(('CEGWM_GEOMETRY_V1_OPERATIONAL_PREFLIGHT ','CEGWM_GEOMETRY_V1_OPERATIONAL_FAILURE ')): raise RuntimeError('unexpected runner receipt')
    payload=json.loads(line.split(' ',1)[1]); receipt_dir=pathlib.Path('/content/geometry-v1-receipt'); receipt_dir.mkdir(exist_ok=False)
    (receipt_dir/'receipt.json').write_text(json.dumps(payload,sort_keys=True,separators=(',',':')),encoding='utf-8')
    terminal='success.json' if proc.returncode == 0 else 'failure.json'; (receipt_dir/terminal).write_text(json.dumps({'status':payload.get('status')},sort_keys=True),encoding='utf-8')
    names={'receipt.json',terminal}; manifest={name:hashlib.sha256((receipt_dir/name).read_bytes()).hexdigest() for name in sorted(names)}
    (receipt_dir/'manifest.json').write_text(json.dumps(manifest,sort_keys=True,separators=(',',':')),encoding='utf-8')
    (receipt_dir/'SHA256SUMS').write_text(''.join(f'{digest}  {name}\n' for name,digest in manifest.items()),encoding='ascii')
    archive=pathlib.Path('/content')/(RUN_ID+'.zip')
    with zipfile.ZipFile(archive,'x',zipfile.ZIP_DEFLATED) as z: [z.write(receipt_dir/name,name) for name in sorted(names|{'manifest.json','SHA256SUMS'})]
    sidecar=archive.with_suffix('.zip.sha256'); sidecar.write_text(hashlib.sha256(archive.read_bytes()).hexdigest()+'  '+archive.name+'\n',encoding='ascii')
    run_dir.mkdir(parents=True); shutil.copy2(archive,run_dir/archive.name); shutil.copy2(sidecar,run_dir/sidecar.name); print(line)
finally:
    child_env.pop('CEG_WM_ROOT_KEY',None); child_env.pop('HF_TOKEN',None); child_env=None
    for path in input_dir.glob('*'): path.unlink()
    input_dir.rmdir()
